# 📑 PageIndex — Vectorless RAG Crash Course
Reasoning-based RAG with No Vector DB, No Chunking

# 🔑 Key Concept
- Traditional RAG → chunk → embed → cosine similarity → retrieve
- PageIndex RAG → build tree → LLM reasons over tree → retrieve exact sections

# 📦 Section 1: Install & Setup

In [3]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [5]:
from pageindex import PageIndexClient
from langchain_groq import ChatGroq

pi_client = PageIndexClient(api_key=os.getenv("PAGEINDEX_API_KEY"))

model = ChatGroq(model="qwen/qwen3-32b")

# 🌲 Section 2: Upload & Index a PDF

## What happens here:

- Upload your PDF to the PageIndex cloud
- PageIndex uses an LLM to read the document structure
- Builds a hierarchical tree index (like a smart Table of Contents)
- Returns a doc_id for all future operations

## Why NO chunking?
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.

In [8]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./data//pdf/1776407131959.pdf"

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ./data//pdf/1776407131959.pdf
✅ Uploaded!
📋 Document ID: pi-cmpzmqgjj002701qxat7sd9mg
   (Save this ID — you'll use it throughout the notebook)


In [9]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: completed

✅ Tree index ready!


# 🔍 Section 3: Inspect the Tree Structure

In [10]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 2

🌲 Raw tree (first node):
{
  "title": "Setup",
  "node_id": "0000",
  "page_index": 1,
  "summary": "# Setup\n",
  "text": "# Setup\n"
}


In [11]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Setup  (p.1)
[0001] OpenClaw with Ollama Locally  (p.1)
  └─ [0002] Wait, what even is OpenClaw?  (p.2)
  └─ [0003] Why go 100% Local?  (p.3)
    └─ [0004] Cloud APIs  (p.3)
    └─ [0005] Local + Ollama  (p.3)
    └─ [0006] The Architecture  (p.4)
    └─ [0007] Prerequisites Checklist  (p.5)
      └─ [0008] Install Ollama &amp; Pull a Model  (p.6)
      └─ [0009] Install Ollama (one command)  (p.6)
      └─ [0010] Install OpenClaw &amp; Run Onboarding  (p.7)
      └─ [0011] Connect &amp; Start Chatting!  (p.8)
      └─ [0012] Start chatting!  (p.8)
      └─ [0013] Which Model Should You Pick?  (p.10)
      └─ [0014] Make It Even Better  (p.11)
  └─ [0015] Your AI. Your Machine. Your Rules.  (p.12)


In [12]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 16
   Each node = one retrievable section of the document


# 🧠 Section 4: LLM Tree Search — The Core of PageIndex

In [24]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model) -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    # response = openai_client.chat.completions.create(
    #     model=model,
    #     messages=[{"role": "user", "content": prompt}],
    #     response_format={"type": "json_object"}
    # )

    response = model.invoke(prompt)

    return json.loads(response.content)

In [25]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is the syllabus covered in Modern LLM finetuning?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree, model)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the syllabus covered in Modern LLM finetuning?



JSONDecodeError: Expecting value: line 1 column 1 (char 0)

# ⚙️ Section 5: Full End-to-End RAG Pipeline